# Этап 1. Свет как волна

Конспект: [[PSI-01-Light-as-Wave]]

Цель — получить словарь, на котором написана статья: **длина волны**, **фаза**, **разность хода**, **волновой фронт**, **интенсивность**.

План:

1. что колеблется и в каких масштабах;
2. бегущая волна — анимация;
3. как пройденный путь превращается в фазу;
4. оптическая длина пути: стекло и воздух;
5. волновой фронт;
6. что на самом деле измеряет камера;
7. комплексная амплитуда.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (8, 3.5)
plt.rcParams["axes.grid"] = True

c = 299_792_458   # скорость света, м/с
lam = 633e-9      # длина волны красного He-Ne лазера, м

## 1. Что колеблется и в каких масштабах

Свет — электромагнитная волна: в каждой точке пространства колеблется электрическое поле $E$. Три величины связаны одной формулой:

$$c = \lambda\,\nu$$

- $\lambda$ — длина волны: расстояние между соседними гребнями;
- $\nu$ — частота: сколько колебаний в секунду;
- $c$ — скорость света.

Видимый свет — это $\lambda$ примерно от 400 нм (фиолетовый) до 700 нм (красный). В интерферометрах чаще всего стоит гелий-неоновый лазер, $\lambda = 632{,}8$ нм.

In [ ]:
nu = c / lam
print(f"λ = {lam * 1e9:.0f} нм")
print(f"частота ν = {nu:.2e} Гц")
print(f"период T = {1 / nu * 1e15:.2f} фс")
print(f"за экспозицию 1 мс поле успевает качнуться {1e-3 * nu:.1e} раз")
print(f"толщина человеческого волоса (~70 мкм) — это {70e-6 / lam:.0f} длин волн")

Два вывода из этих чисел:

- длина волны — готовая **линейка** с делением около полумикрона, поэтому интерферометрия и даёт нанометровую точность;
- колебания слишком быстрые, чтобы какой-либо приёмник мог за ними уследить (к этому вернёмся в разделе 6).

## 2. Бегущая волна

$$E(z, t) = E_0\cos\!\left(\frac{2\pi}{\lambda}z - 2\pi\nu t + \varphi_0\right)$$

Всё выражение в скобках — **фаза** волны в точке $z$ в момент $t$. У этой формулы два «лица»:

- **заморозим время** — получим косинусоиду в пространстве с периодом $\lambda$;
- **встанем в одну точку** — увидим колебание во времени с периодом $T = 1/\nu$, ровно как в этапе 0.

Знак минус перед $t$ означает, что волна бежит в сторону роста $z$.

In [ ]:
z = np.linspace(0, 4, 400)             # координата в длинах волн
frames = 60
times = np.arange(frames) / 30         # время в периодах: два периода на анимацию
z0 = 2.5                               # точка наблюдения

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [2, 1]})
wave, = ax1.plot([], [], "C0")
crest, = ax1.plot([], [], "kv", ms=9, label="гребень")
probe, = ax1.plot([], [], "ro", ms=8, label="точка наблюдения")
ax1.axvline(z0, color="r", lw=0.5)
ax1.set_xlim(0, 4)
ax1.set_ylim(-1.4, 1.4)
ax1.set_xlabel("z, длины волн")
ax1.set_ylabel("поле E")
ax1.set_title("снимок волны в пространстве")
ax1.legend(loc="upper right", ncol=2)

ax2.plot(times, np.cos(2 * np.pi * (z0 - times)), "0.8")
trace, = ax2.plot([], [], "r")
ax2.set_xlabel("t, периоды")
ax2.set_title("колебание в точке наблюдения")


def update(i):
    t = times[i]
    wave.set_data(z, np.cos(2 * np.pi * (z - t)))
    crest.set_data([1 + t], [1.12])            # гребень смещается на λ за один период
    probe.set_data([z0], [np.cos(2 * np.pi * (z0 - t))])
    trace.set_data(times[: i + 1], np.cos(2 * np.pi * (z0 - times[: i + 1])))
    return wave, crest, probe, trace


anim = FuncAnimation(fig, update, frames=frames, interval=60, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

Гребень за один период проходит ровно одну длину волны. Красная точка при этом никуда не едет — она только качается вверх-вниз.

## 3. Путь превращается в фазу

Сравним волну в двух точках, разнесённых на расстояние $\Delta$. Вторая отстаёт по фазе на

$$\varphi = \frac{2\pi}{\lambda}\,\Delta$$

Это главная формула интерферометрии. Она переводит расстояния в углы:

| Разность хода $\Delta$ | Разность фаз | Для $\lambda$ = 633 нм |
|---|---|---|
| $\lambda$ | 360° | 633 нм |
| $\lambda/2$ | 180° | 316 нм |
| $\lambda/4$ | 90° | 158 нм |
| $\lambda/360$ | 1° | 1,8 нм |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, (frac, name) in zip(axes.flat, [(0, "0"), (0.25, "λ/4"), (0.5, "λ/2"), (1.0, "λ")]):
    ax.plot(z, np.cos(2 * np.pi * z), "C0", label="волна 1")
    ax.plot(z, np.cos(2 * np.pi * (z - frac)), "C1", label="волна 2 (прошла путь на Δ больше)")
    ax.set_title(f"Δ = {name}  →  разность фаз {frac * 360:.0f}°", fontsize=10)
axes[0, 0].legend(fontsize=7, loc="lower left")
for ax in axes[1]:
    ax.set_xlabel("z, длины волн")
plt.tight_layout()
plt.show()

При $\Delta = \lambda$ волны снова неразличимы. Отсюда неоднозначность, которая потом всплывёт как «несшитая фаза»: по самим волнам нельзя отличить разность хода $0{,}1\lambda$ от $1{,}1\lambda$.

## 4. Оптическая длина пути

В прозрачной среде свет идёт медленнее в $n$ раз ($n$ — показатель преломления). Частота не меняется, поэтому длина волны укорачивается до $\lambda/n$, и на том же отрезке укладывается больше колебаний. Фазу определяет не геометрический путь $d$, а **оптический**:

$$L = n\,d, \qquad \varphi = \frac{2\pi}{\lambda}\,n\,d$$

Пластинка толщиной $d$ вместо воздуха добавляет разность хода $(n-1)\,d$.

In [ ]:
n_glass = 1.5
z = np.linspace(0, 10, 2000)                 # координата в длинах волн (в вакууме)
slab = (z >= 4) & (z <= 6)                   # стеклянная пластинка толщиной 3λ

n_profile = np.where(slab, n_glass, 1.0)
optical_path = np.cumsum(n_profile) * (z[1] - z[0])   # ∫ n dz

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axvspan(4, 6, color="C0", alpha=0.15, label=f"стекло, n = {n_glass}")
ax.plot(z, np.cos(2 * np.pi * z), "0.6", label="волна мимо пластинки")
ax.plot(z, np.cos(2 * np.pi * optical_path), "C3", label="волна сквозь пластинку")
ax.set_xlabel("z, длины волн")
ax.set_ylabel("поле E")
ax.legend(loc="upper right", ncol=3, fontsize=8)
ax.set_ylim(-1.2, 1.6)
plt.show()

extra = (n_glass - 1) * 2
print(f"добавочная разность хода (n − 1)·d = {extra} λ  →  разность фаз {extra * 360:.0f}°")

Внутри стекла гребни идут чаще. После пластинки волна отстаёт на полторы длины волны — она оказалась в противофазе с той, что шла мимо.

То же самое, только слабее, делает **воздух**. Его показатель преломления зависит от температуры — примерно на $10^{-6}$ на каждый градус.

In [ ]:
dn_dT = 0.93e-6        # изменение показателя преломления воздуха на 1 °C (видимый свет, комнатные условия)
path = 0.1             # длина пути луча в воздухе, м

for dT in (0.01, 0.5, 1.0):
    opd = dn_dT * dT * path
    print(f"ΔT = {dT:<4} °C на пути {path} м  →  разность хода {opd * 1e9:6.1f} нм  =  {opd / lam * 360:6.1f}° фазы")

Тёплая струя воздуха с перепадом в десятую долю градуса на метровом пути сдвигает фазу на полсотни градусов. Это один из источников тех самых «неизвестных сдвигов», с которыми борется статья.

## 5. Волновой фронт

До сих пор волна была одномерной. В пространстве важно, **где** фаза одинакова.

**Волновой фронт** — поверхность, во всех точках которой фаза волны одна и та же (например, поверхность одного гребня). Свет распространяется перпендикулярно фронту.

- **Плоская волна** — фронты-плоскости. Так выглядит луч лазера после расширителя пучка.
- **Сферическая волна** — фронты-сферы вокруг точечного источника.

In [ ]:
grid = np.linspace(-5, 5, 300)               # координаты в длинах волн
X, Z = np.meshgrid(grid, grid)

angle = np.deg2rad(20)
phase_plane = 2 * np.pi * (Z * np.cos(angle) + X * np.sin(angle))
phase_sphere = 2 * np.pi * np.hypot(X, Z + 5)        # точечный источник внизу по центру

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6))
kw = dict(cmap="RdBu", vmin=-1, vmax=1, extent=[-5, 5, -5, 5], origin="lower")
im1 = ax1.imshow(np.cos(phase_plane), **kw)
im2 = ax2.imshow(np.cos(phase_sphere), **kw)
ax1.set_title("плоская волна (идёт под углом 20°)")
ax2.set_title("сферическая волна")
for ax in (ax1, ax2):
    ax.grid(False)
    ax.set_xlabel("x, длины волн")
ax1.set_ylabel("z, длины волн")

frames = 30


def update(i):
    shift = 2 * np.pi * i / frames
    im1.set_data(np.cos(phase_plane - shift))
    im2.set_data(np.cos(phase_sphere - shift))
    return im1, im2


anim = FuncAnimation(fig, update, frames=frames, interval=60, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

Красные полосы — гребни, синие — впадины. Каждая красная линия — это один волновой фронт.

### Искажённый фронт

Идеальное плоское зеркало отражает плоскую волну в плоскую. Если на зеркале есть бугорок высотой $h$, свет, попавший на него, проходит путь на $2h$ короче — туда и обратно. Фронт в этом месте **выпячивается вперёд на $2h$**.

Отклонение реального фронта от идеального описывают функцией $W(x, y)$ — её измеряют в длинах волн. Фаза связана с ней всё той же формулой:

$$\varphi(x, y) = \frac{2\pi}{\lambda}\,W(x, y)$$

Именно $W(x,y)$ и есть конечный результат всей статьи — «топограмма волнового фронта».

In [ ]:
x = np.linspace(-5, 5, 300)
zz = np.linspace(0, 6, 240)
Xb, Zb = np.meshgrid(x, zz)

h = 0.15                                    # высота бугорка на зеркале, в длинах волн
W = 2 * h * np.exp(-x**2 / 2)               # искажение фронта после отражения: вдвое больше

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.4, 1]})
ax1.imshow(np.cos(2 * np.pi * (Zb - W[None, :])), cmap="RdBu", vmin=-1, vmax=1,
           extent=[-5, 5, 0, 6], origin="lower", aspect="auto")
ax1.plot(x, 3 + W, "k", lw=2, label="один волновой фронт")
ax1.grid(False)
ax1.set_xlabel("x, длины волн")
ax1.set_ylabel("z, длины волн")
ax1.set_title("отражённая волна (идёт вверх)")
ax1.legend(loc="upper right")

ax2.plot(x, W, "k", label="искажение фронта W(x)")
ax2.plot(x, W / 2, "C1--", label="бугорок на зеркале h(x)")
ax2.set_xlabel("x, длины волн")
ax2.set_ylabel("длины волн")
ax2.legend(fontsize=8)
plt.show()

print(f"бугорок {h} λ = {h * lam * 1e9:.0f} нм  →  искажение фронта {2 * h} λ  →  фаза {2 * h * 360:.0f}°")

Бугорок высотой 95 нм — в сотни раз тоньше волоса — даёт сдвиг фазы больше 100°. Вопрос только в том, как эту фазу увидеть.

## 6. Что измеряет камера

Любой приёмник света (глаз, фотодиод, пиксель камеры) реагирует на **энергию**, накопленную за время экспозиции. Мгновенная мощность пропорциональна $E^2$, а приёмник выдаёт её среднее:

$$I \propto \langle E^2\rangle = \frac{E_0^2}{2}$$

Посмотрим, как быстро устанавливается это среднее и зависит ли оно от фазы.

In [ ]:
t = np.linspace(0, 6, 3000)                  # время в периодах
E0 = 1.0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for phi_deg, color in ((0, "C0"), (120, "C1")):
    E = E0 * np.cos(2 * np.pi * t + np.deg2rad(phi_deg))
    running_mean = np.cumsum(E**2) / np.arange(1, t.size + 1)
    ax1.plot(t, E, color, label=f"E(t), фаза {phi_deg}°")
    ax2.plot(t, E**2, color, lw=0.6, alpha=0.5)
    ax2.plot(t, running_mean, color, lw=2, label=f"накопленное среднее, фаза {phi_deg}°")
ax2.axhline(E0**2 / 2, color="k", ls="--", lw=1, label="E₀²/2")
ax1.set_ylabel("поле")
ax1.legend(loc="upper right", ncol=2, fontsize=8)
ax2.set_ylabel("мощность E²")
ax2.set_xlabel("t, периоды")
ax2.legend(loc="upper right", ncol=3, fontsize=8)
plt.show()

Уже через несколько периодов среднее неотличимо от $E_0^2/2$ — а за реальную экспозицию проходит порядка $10^{11}$ периодов. Обе кривые сходятся к одному числу: **в интенсивности одиночной волны информации о фазе нет**.

Получается противоречие. Форма поверхности записана в фазе $\varphi(x,y)$ (раздел 5), а камера фазу не видит. Выход — сложить исследуемую волну с другой, опорной: тогда яркость начинает зависеть от **разности** их фаз. Это и есть интерференция, этап 2.

## 7. Комплексная амплитуда

Раз за колебанием во времени всё равно не уследить, множитель $e^{-i\omega t}$ отбрасывают и описывают волну в данной точке одним комплексным числом:

$$U = A\,e^{i\varphi}, \qquad I = |U|^2$$

Это вектор из этапа 0, «сфотографированный» в какой-то момент: длина — амплитуда, угол — фаза. Множитель $1/2$ прячут в определение $A$.

In [ ]:
A = 1.3
fig, ax = plt.subplots(figsize=(4.5, 4.5))
circle = np.linspace(0, 2 * np.pi, 200)
ax.plot(A * np.cos(circle), A * np.sin(circle), "0.8")

for phi_deg in (0, 60, 120, 250):
    U = A * np.exp(1j * np.deg2rad(phi_deg))
    ax.plot([0, U.real], [0, U.imag], "o-", label=f"φ = {phi_deg}°")
    print(f"φ = {phi_deg:3d}°   U = {U.real:+.3f} {U.imag:+.3f}i   |U|² = {abs(U)**2:.3f}")

ax.set_aspect("equal")
ax.set_xlabel("Re U")
ax.set_ylabel("Im U")
ax.legend(fontsize=8)
plt.show()

Четыре разных вектора, четыре разных фазы — и одна и та же интенсивность $|U|^2$. Зато складывать волны теперь просто: складываем векторы. Длина суммы зависит от угла между ними — с этого начинается этап 2.

## Итог

| Понятие | Что это | Формула |
|---|---|---|
| Длина волны $\lambda$ | расстояние между гребнями; «деление линейки» | $c = \lambda\nu$ |
| Фаза | где мы находимся внутри периода колебания | $\varphi = \dfrac{2\pi}{\lambda}\Delta$ |
| Оптическая длина пути | путь с учётом замедления в среде | $L = n\,d$ |
| Волновой фронт | поверхность равной фазы | $\varphi(x,y) = \dfrac{2\pi}{\lambda}W(x,y)$ |
| Интенсивность | то, что видит камера | $I = \lvert U\rvert^2$ |

## Упражнения

1. На сколько нанометров нужно сдвинуть зеркало, чтобы отражённая от него волна изменила фазу на 90°? (Не забудьте про «туда и обратно».) Сравните с таблицей из ответа про пьезопривод.
2. В разделе 4 подберите толщину пластинки так, чтобы волна после неё совпала по фазе с волной мимо пластинки. Какая минимальная толщина для $n = 1{,}5$?
3. Пятно тёплого воздуха ($\Delta T = 0{,}5$ °C) шириной 10 см пересекает луч. Какой сдвиг фазы оно даёт? Заметен ли он, если нужна точность $\lambda/100$?
4. В разделе 5 замените бугорок на наклон: `W = 0.1 * x`. Как выглядят фронты? Чем наклон зеркала отличается от его смещения вдоль луча?
5. Сложите два комплексных числа одинаковой длины с углами 0° и 120° и посчитайте $|U_1 + U_2|^2$. Сравните с $|U_1|^2 + |U_2|^2$. Повторите для 0° и 180°.

In [ ]:
#1
d_phi = 90
h = d_phi/(2 * 360)*lam

print(h)

U1 = 1 * np.exp(1j * np.deg2rad(0))
U2 = 1 * np.exp(1j * np.deg2rad(90))

print("сумма полей:        ", abs(U1 + U2)**2)
print("сумма интенсивностей:", abs(U1)**2 + abs(U2)**2)

